## Imports

In [ ]:
# Deep copy method from standard library for copying a "template" dictionary
from copy import deepcopy

# Scientific libraries
import numpy as np

# File editing/parsing libraries
import os
from pathlib import Path
from scipy.io import savemat
from lxml import etree

# Visualization libraries
import mediapy as media
from dm_control.mujoco.wrapper.core import MjvOption
import matplotlib.pyplot as plt
import matplotlib.animation as pltAnim
%matplotlib inline

# Policy libraries
import tensorflow as tf
from acme import wrappers

# Flybody, and its relevant classes/functions/constants
import flybody
from flybody.agents.utils_tf import TestPolicyWrapper
from flybody.download_data  import figshare_download
from flybody.fly_envs       import flight_imitation
from flybody.tasks.constants import _FLY_PHYSICS_TIMESTEP, _FLY_CONTROL_TIMESTEP, _BODY_PITCH_ANGLE
from flybody.tasks.synthetic_trajectories import constant_speed_trajectory
from flybody.quaternions    import rotate_vec_with_quat,reciprocal_quat


# Prevent tensorflow from stealing all gpu memory.
physical_devices = tf.config.list_physical_devices('GPU')
print("physical devices:",physical_devices)
for device in physical_devices:
    tf.config.experimental.set_memory_growth(device, True)

### Initialization
Users: Customize simulation parameters here. For the simulation to work properly, do not modify any other part of the code (unless you know what you are doing)

In [ ]:
# FLIGHT_IMITATION_PATH   = '/home/exj5139/Downloads/'+'datasets_flight-imitation/'   # add the folder path to the supplementry dataset
# POLICY_PATH             = '/home/exj5139/Downloads/'+'trained-fly-policies/'        # add the folder path to the supplementry dataset
# LOADSAVE_PATH           = '/home/exj5139/Downloads/'+'ablation_results.mat'         # the path you want to save to or load from

# USER INPUT
# "SAVE_DIR"      : the directory where the results, datasets, and policies are stored. Creates a new directory if it doesn't already exist
# "SAVE_RESULTS"  : if true, the numerical simulation data will be stored as a .mat file, figures as .svg, and videos as .mp4
# "T_TOTAL"       : total simulation time (s): default is an arbitary time 
# "FLY_SPEED"     : the speed of the trajectory that the fruitfly is to keep up with (cm/s). It is up to the user to keep this number reasonable 
# "WAR_INCR"      : the amount of wing area ratio to increase between simulation episodes. The simulation time and result file size will grow if this is lowered
# "CAMERA"        : set to an integer that corresponds to a MjCamera in Flybody: this will change the viewing angle of the video, if taken
# "RECORD_VIDEOS" : set True if you want to record videos of the first, middle, and last episodes, both closed-loop and open-loop 
# "VERBOSE"       : if True, prints runtime information

UI = {
    "SAVE_DIR"      : "",
    "SAVE_RESULTS"  : False,
    "T_TOTAL"       : 0.08,
    "FLY_SPEED"     : 20.0,
    "WAR_INCR"      : 0.10,
    "CAMERA"        : 4,
    "RECORD_VIDEOS" : True,
    "VERBOSE"       : True
}

### Definitions

In [ ]:
## fruitfly xml files
ff_source_path = flybody.__path__[0]+'/fruitfly/build_fruitfly/fruitfly.xml'
ff_working_path= flybody.__path__[0]+'/fruitfly/assets/fruitfly.xml'

# Rendering options
scene_opt = MjvOption()
scene_opt.geomgroup = [1,0,0,1,1,0] # changing this will change the fly's appearance as captured by a MjCamera
RENDER_KWARGS = {'width': 640, 'height': 480, 'scene_option': scene_opt}
videos = [[],[],[]]

# Allocate.
N_STEPS = int(UI["T_TOTAL"] /_FLY_PHYSICS_TIMESTEP) # frames
wars = np.flip(np.arange(0.5,1+0.5*UI["WAR_INCR"],UI["WAR_INCR"]))
M_STEPS = wars.size
ROOT_QPOS0 = np.array([0,0,1,np.cos(np.deg2rad(_BODY_PITCH_ANGLE)*0.5),0,-np.sin(np.deg2rad(_BODY_PITCH_ANGLE)*0.5),0])

# Simulation state save
flybody_states = { # N_STEPS by M_STEPS = time vs damage
    "warline"   : wars,                                     # wing area ratio axis
    "timeline"  : _FLY_CONTROL_TIMESTEP*np.arange(N_STEPS), # time axis
    "bodyForces": np.nan * np.zeros((N_STEPS, M_STEPS, 6)), # 6 DOFs (Fx,Fy,Fz,Tx,Ty,Tz) NOTE: forces are in GLOBAL FRAME while torques are in LOCAL FRAME
    "bodyPose": {
        "root"      : np.nan * np.zeros((N_STEPS, M_STEPS, 7)), # thorax displ + rot (x,y,z,w,wx,wy,wz) in the GLOBAL FRAME
        "head"      : np.nan * np.zeros((N_STEPS, M_STEPS, 3)), # head angles (alpha,beta,gamma) in the HEAD LOCAL FRAME
        "wing"      : np.nan * np.zeros((N_STEPS, M_STEPS, 6)), # wing angles (stroke,deviation/pitch,rotation) in the THORAX LOCAL FRAME
        "abdomen"   : np.nan * np.zeros((N_STEPS, M_STEPS,4))   # the abdomen vector and the angle which it forms against the y-plane within the THORAX LOCAL FRAME 
    },
    "action": {                                                 # [-1,1] normalized commands for the actuated joints relevant to flying
        "head"      : np.nan * np.zeros((N_STEPS, M_STEPS, 3)),
        "wing"      : np.nan * np.zeros((N_STEPS, M_STEPS, 6)),
        "abdomen"   : np.nan * np.zeros((N_STEPS, M_STEPS, 2)),
        "freqMod"   : np.nan *np.zeros((N_STEPS, M_STEPS, 1))   # this isn't an actuated joint: it controls how much the wingbeat frequency deviates from 218Hz
    },
    "acceleration": np.nan * np.zeros((N_STEPS,M_STEPS,3)),     # also capture the root linear acceleration
    "userSettings": UI
}

results = {"cloop":deepcopy(flybody_states), "oloop":deepcopy(flybody_states)}
print("total timesteps:",N_STEPS,"\ntotal warsteps: ",M_STEPS)

def plot(save_dir,ctrl_ts, joints_pos, actions, root_qpos):
    time_axis = np.arange(N_STEPS) * ctrl_ts * 1000  # ms
    joint_labels = []
    for i in range(7):
        joint_labels.append(f"abduct{i}")
        joint_labels.append(f"extend{i}")
    
    # Proprioception plot
    plt.figure(figsize=(6, 10))
    plt.suptitle('Proprioception sensory inputs: head, abdomen, wing joint angles')
    plt.subplot(4, 1, 1)  # Head joints.
    plt.plot(time_axis, joints_pos[:, :3], label=['head_abduct', 'head_twist', 'head'])
    plt.ylabel('Head joint angles (rad)')
    plt.legend()
    plt.subplot(4, 1, 2)  # Abdomen joints.
    plt.plot(time_axis, joints_pos[:, 9:23], label=joint_labels)
    plt.ylabel('Abdomen joint angles (rad)')
    plt.subplot(4, 1, 3)  # Left wing.
    plt.plot(time_axis, joints_pos[:, 3:6], label=['yaw', 'roll', 'pitch'])
    plt.ylabel('Left wing angles (rad)')
    plt.legend()
    plt.subplot(4, 1, 4)  # Right wing.
    plt.plot(time_axis, joints_pos[:, 6:9])
    plt.xlabel('Time (ms)')
    plt.ylabel('Right wing angles (rad)')
    plt.tight_layout()
    plt.savefig(save_dir+r"proprioception.svg")

    # Dimensionless action specs
    plt.figure(figsize=(6,10))
    plt.subplot(2, 1, 1)
    plt.plot(time_axis, actions[:, :3], label=['yaw', 'roll', 'pitch'])
    plt.ylabel('Left wing control\n(unitless)')
    plt.legend()
    plt.subplot(2, 1, 2)
    plt.plot(time_axis, actions[:, 3:])
    plt.xlabel('Time (ms)')
    plt.ylabel('Right wing control')
    plt.tight_layout()
    plt.savefig(save_dir+r"wing_actions.svg")

    # Position
    plt.figure(figsize=(6,10))
    plt.subplot(2, 1, 1)
    plt.plot(time_axis, root_qpos[:, :3], label=['x', 'y', 'z'])
    plt.ylabel('Global Position (cm)')
    plt.legend()
    plt.subplot(2, 1, 2)
    plt.plot(time_axis, root_qpos[:, 3:], label=['w', 'x', 'y', 'z'])
    plt.xlabel('Time (ms)')
    plt.ylabel('Global Quaternion')
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_dir+r"pose.svg")


def CutRightWing(ff_source_path: str, ff_working_path: str, war: float):
    """
    Apply a unilateral chordwise cut to the right wing of a fruitfly.xml file.
    
    :param ff_source_path: The local path of the fruitfly.xml file to READ from, ideally a copy of the original file.
    :type ff_source_path: str
    :param ff_working_path: The local path of the fruitfly.xml to WRITE to. Expected to be the location from which an Environment is generated from 
    :type ff_working_path: str
    :param war: Wing Area Ratio. The amount of wing that is left after being damaged. Ranges from (0,1]
    :type war: float
    """
    #with open(ff_source_path,'r') as f: tree = etree.XML(f.read(), etree.XMLParser(remove_blank_text=True))
    tree = etree.parse(ff_source_path)
    
    fluid   = tree.xpath("//geom[@name=\"wing_right_fluid\"]")[0]
    inertial= tree.xpath("//geom[@name=\"wing_right_inertial\"]")[0]

    # Edit wing length
    size0 = fluid.attrib["size"].split(" ") # get initial length as a list of floatlike strings
    size0 = np.array([float(size_i) for size_i in size0]) # make it an array of floats
    len0 = size0[2] # save this value, used later
    size1 = np.array([1,1,war])*size0 # calculate new size
    len1 =  size1[2] # save this too
    size1 = ' '.join(str(size_i) for size_i in size1) # convert back into a space-delimited listlike string

    # Edit wing position
    pos0 = fluid.attrib["pos"].split(" ") # get initial position as a list of floatlike strings
    pos0 = np.array([float(pos_i) for pos_i in pos0]) # convert to array of floats
    pos1 = pos0 + np.array([0,len1-len0,0]) # calculate new position
    pos1 = ' '.join(str(pos_i) for pos_i in pos1) # convert into space-delimited listlike string

    # Add edits to tree
    for geom in [fluid, inertial]:
        geom.attrib["size"] = size1
        geom.attrib["pos"] = pos1
    inertial.attrib["mass"] = str(8e-6 * war) # also set the new mass of inertia

    # Add force/torque sensors
    f_sensor = etree.SubElement(tree.find("sensor"), "force")
    f_sensor.set("name","force_thorax"); f_sensor.set("site","thorax")
    t_sensor = etree.SubElement(tree.find("sensor"), "torque")
    t_sensor.set("name","torque_thorax"); t_sensor.set("site","thorax")

    # Finally, save all edits
    tree.write(ff_working_path)

def initSimulationFolder() -> Path:
    # Verify that parent path exists
    parent_dir = Path(UI["SAVE_DIR"]).absolute()
    if not parent_dir.exists():
        if UI["VERBOSE"]: print(f"SAVE_DIR={UI['SAVE_DIR']} not found: defaulted to current directory")
        parent_dir = Path.cwd().absolute()
    
    # Folder structure
    main_dir    = parent_dir / "ablation-simulation"
    dataset_dir = main_dir / "flight-imitation-datasets"
    policy_dir  = main_dir / "trained-fly-policies"
    results_dir = main_dir / "results"

    # Check if the simulation folder already exists
    if not main_dir.exists():
        if UI["VERBOSE"]: print("Previous save data not found, new simulation folder created and downloading datasets...",end=" ")
        main_dir.mkdir()
        dataset_dir.mkdir()
        figshare_download("flight-imitation-dataset",dataset_dir.as_posix())
        policy_dir.mkdir()
        figshare_download("trained-fly-policies",policy_dir.as_posix())
        results_dir.mkdir()
        if UI["VERBOSE"]: print("complete")
    else: # Verify that the subdirectories all exists, just in case something changed or got deleted.
        if not dataset_dir.exists():
            if UI["VERBOSE"]: 
                print("flight-imitation-datasets not found in simulation folder: downloading...",end=" ")
                figshare_download("flight-imitation-datasets",dataset_dir.as_posix())
                print("complete")
            else: figshare_download("flight-imitation-datasets",dataset_dir.as_posix())
        if not policy_dir.exists():
            if UI["VERBOSE"]: 
                print("trained-fly-policies not found in simulation folder: downloading...",end=" ")
                figshare_download("trained-policies",policy_dir.as_posix())
                print("complete")
            else: figshare_download("trained-policies",policy_dir.as_posix())
        if not results_dir.exist():
            if UI["VERBOSE"]: print("results folder not found: generated instead")
            results_dir.mkdir()
    return main_dir

### Generate Results
Either by simulation or loading from a mat file

In [ ]:
## SIMULATE
MAIN_PATH = initSimulationFolder()
WPG_PATTERN_PATH = MAIN_PATH/"flight-imitation-datasets"/"wing_pattern_fmech.npy"
FLIGHT_POLICY_PATH = MAIN_PATH/"trained-fly-policies"/"flight"

for m,war in enumerate(wars):
    # SIMULATION INITIALIZATION ###############################################################################################
    CutRightWing(ff_source_path,ff_working_path,war) # Edit model with the appropriate WAR BEFORE loading the environment
    env = flight_imitation(None, WPG_PATTERN_PATH.as_posix(), terminal_com_dist=float('inf')) # initialize env with no trajectory...
    qpos, qvel = constant_speed_trajectory( #... and set the trajectory here
                n_steps=N_STEPS, speed=UI["FLY_SPEED"], init_pos=(0, 0, 1),
                body_rot_angle_y=-47.5, control_timestep=_FLY_CONTROL_TIMESTEP)
    env.task._traj_generator.set_next_trajectory(qpos, qvel)

    # Wrap environment and policy
    env = wrappers.SinglePrecisionWrapper(env)
    env = wrappers.CanonicalSpecWrapper(env, clip=True)
    flight_policy = tf.saved_model.load(FLIGHT_POLICY_PATH)
    flight_policy = TestPolicyWrapper(flight_policy)

    # Get normalization factors
    ref_force = np.abs(
        env.physics.model.opt.gravity[2]*env.physics.model.body_subtreemass[2]
    ) # mass and gravity (ie the fly weight) normalizes force
    ref_torque= np.abs(
        env.physics.model.opt.gravity[2]*env.physics.model.body_subtreemass[2]*0.114
    ) # weight and characteristic length (undamaged wing chord length) normalizes torque
    #########################################################################################################################

    # CLOSED-LOOP SIMULATION ################################################################################################
    # Follow the defined trajectory with the flight policy, accepting sensory feedback as input
    timestep = env.reset()
    n = 0

    while(timestep.step_type != 2):
        # Record some of the sensory inputs.
        
        # Body Pose: get pose data of fly
        pos = env.physics.data.qpos[:3].copy()
        quat= env.physics.data.qpos[3:7].copy()
        results["cloop"]["bodyPose"]["root"][n,m,:3 ] = pos
        results["cloop"]["bodyPose"]["root"][n,m,3: ] = quat
        results["cloop"]["bodyPose"]["head"][n,m    ] = timestep.observation['walker/joints_pos'][0:3]
        results["cloop"]["bodyPose"]["wing"][n,m    ] = timestep.observation['walker/joints_pos'][3:9]

        # The abdomen angle can't be directly measured: it's the net transformation of all the individual abdomen body elements.
        # What we can do instead is take the vector between the abdomen base and tip within the fly's body(thorax) frame. We can
        # then expresss the angle as the angle between the vector and the y-plane. This shows how much the abdomen deviates to 
        # the side
        fly_frame       = env.physics.data.xquat[1] # the thorax coordinate frame
        abdomen_base_pos= env.physics.data.xpos[11] # the abdomen base global position.
        abdomen_tip_pos = env.physics.data.xpos[17] # the abdomen tip global position
        abdomen_vec     = rotate_vec_with_quat(abdomen_tip_pos-abdomen_base_pos,reciprocal_quat(fly_frame)) # vectorize the base to tip direction
        abdomen_angle   = np.rad2deg(np.arcsin(abdomen_vec[1]/np.linalg.norm(abdomen_vec)))
        results["cloop"]["bodyPose"]["abdomen"][n,m] = [*abdomen_vec,abdomen_angle]


        # Body Forces and acceleration
        forces  = env.physics.data.sensordata[9:12] / ref_force 
        forces  = rotate_vec_with_quat(forces,quat) # rotate into global frame
        torques = env.physics.data.sensordata[12:15] / ref_torque
        results["cloop"]["bodyForces"][n,m,:3] = forces
        results["cloop"]["bodyForces"][n,m,3:]= torques
        results["cloop"]["acceleration"][n,m]=rotate_vec_with_quat(timestep.observation["walker/accelerometer"],quat)

        # maybe capture a frame
        if UI["RECORD_VIDEOS"]:
            if war==wars[0]: # beginning
                videos[0].append(env.physics.render(camera_id=UI["CAMERA"], **RENDER_KWARGS))
            elif war==wars[int(wars.size*0.5)]: # middle
                videos[1].append(env.physics.render(camera_id=UI["CAMERA"], **RENDER_KWARGS))
            elif war==wars[-1]: # end
                videos[2].append(env.physics.render(camera_id=UI["CAMERA"], **RENDER_KWARGS))

        # Advance simulation (by generating action)
        action = flight_policy(timestep.observation)
        timestep = env.step(action)

        # Record wing action commands.
        results["cloop"]["action"]["head"   ][n,m] = action[:3]
        results["cloop"]["action"]["wing"   ][n,m] = action[3:9]
        results["cloop"]["action"]["abdomen"][n,m] = action[9:11]
        results["cloop"]["action"]["freqMod"][n,m] = action[11]

        # Increment the step
        n += 1
    #######################################################################################################################

    # OPEN-LOOP SIMULATION ################################################################################################
    # Replay the resultant wing kinematics from the closed-loop simulation on a rigidly tethered fly
    timestep = env.reset()
    final_n = n # sometimes N_STEPS isn't exactly when the simulation ends, so let's save the last recorded n value 
    n = 0
    undamaged_key = list(results["cloop"].keys())[-1]

    while(n < final_n and timestep.step_type != 2):
        # Record some of the sensory inputs.
        
        # Body Pose: get pose data of fly
        pos = env.physics.data.qpos[:3].copy()
        quat= env.physics.data.qpos[3:7].copy()
        results["cloop"]["bodyPose"]["root"][n,m,:3 ] = pos
        results["cloop"]["bodyPose"]["root"][n,m,3: ] = quat
        results["cloop"]["bodyPose"]["head"][n,m    ] = timestep.observation['walker/joints_pos'][0:3]
        results["cloop"]["bodyPose"]["wing"][n,m    ] = timestep.observation['walker/joints_pos'][3:9]

        # The abdomen angle can't be directly measured: it's the net transformation of all the individual abdomen body elements.
        # What we can do instead is take the vector between the abdomen base and tip within the fly's body(thorax) frame. We can
        # then expresss the angle as the angle between the vector and the y-plane. This shows how much the abdomen deviates to 
        # the side
        fly_frame       = env.physics.data.xquat[1,:] # the thorax coordinate frame
        abdomen_base_pos= env.physics.data.xpos[11,:] # the abdomen base global position.
        abdomen_tip_pos = env.physics.data.xpos[17,:] # the abdomen tip global position
        abdomen_vec     = rotate_vec_with_quat(abdomen_tip_pos-abdomen_base_pos,reciprocal_quat(fly_frame)) # vectorize the base to tip direction
        abdomen_angle   = np.rad2deg(
            0.5*np.pi - np.arccos(abdomen_vec[1]/np.linalg.norm(abdomen_vec))
        )
        results["cloop"]["bodyPose"]["abdomen"][n,m] = [*abdomen_vec,abdomen_angle]


        # Body Forces and acceleration
        forces  = env.physics.data.sensordata[9:12] / ref_force 
        forces  = rotate_vec_with_quat(forces,quat) # rotate into global frame
        torques = env.physics.data.sensordata[12:15] / ref_torque
        results["cloop"]["bodyForces"][n,m,:3] = forces
        results["cloop"]["bodyForces"][n,m,3:]= torques
        results["cloop"]["acceleration"][n,m]=rotate_vec_with_quat(timestep.observation["walker/accelerometer"],quat)

        # maybe capture a frame
        if UI["RECORD_VIDEOS"]:
            if war==wars[0]: # beginning
                videos[0].append(env.physics.render(camera_id=UI["CAMERA"], **RENDER_KWARGS))
            elif war==wars[int(wars.size*0.5)]: # middle
                videos[1].append(env.physics.render(camera_id=UI["CAMERA"], **RENDER_KWARGS))
            elif war==wars[-1]: # end
                videos[2].append(env.physics.render(camera_id=UI["CAMERA"], **RENDER_KWARGS))

        # Advance simulation (by reading the  closed loop, undamaged actions)
        action = np.array([
            *results["cloop"]["action"]["head"   ][n,0],
            *results["cloop"]["action"]["wing"   ][n,0],
            *results["cloop"]["action"]["abdomen"][n,0],
            *results["cloop"]["action"]["freqMod"][n,0]])
        timestep = env.step(action)

        # Increment the step
        n += 1
    #######################################################################################################################

# POST_PROCESS ##################################################################################################################
# Since we edited the assets/flybody.xml to cut the wings, let's revert the changes by copying the contents of build_fruitfly.xml
etree.parse(ff_source_path).write(ff_working_path)

# Maybe save results
if UI["SAVE_RESULTS"]:
    if UI["VERBOSE"]: print("saving results")
    savemat(UI["RESULT_DIR"], results)

# Maybe show the videos
if UI["RECORD_VIDEOS"]: 
    if UI["VERBOSE"]: print("showing videos")
    media.show_videos(videos)
#################################################################################################################################

## Plottting

In [ ]:
fig = plt.figure(figsize=plt.figaspect(2.))
fig.suptitle("Abdomen Tracking (WAR=0.75)")
t = results["cloop"]["timeline"]
x = results["cloop"]["bodyPose"]["abdomen"][:,int(M_STEPS/2),:]

# First subplot, abdomen angle
ax = fig.add_subplot(2,1,1)
oneD   = ax.plot(t[0],x[0,3])
ax.set(xlim=[0,np.max(t)], ylim=[-90,90])
ax.grid(True)
ax.set_ylabel("Abdomen Angle, deg")

# Second subplot, 3D reconstruction
ax = fig.add_subplot(2,1,2,projection='3d')
threeD = ax.plot([0,x[0,0]],[0,x[0,1]],[0,x[0,2]])
absBound = [np.nanmin(x[:,:3]),np.nanmax(x[:,:3])]
ax.set(xlim=absBound, ylim=absBound, zlim=absBound)
ax.set_xlabel("body axis")

def update(frame):
    # for each frame...
    oneD[0].set_xdata(t[:frame])
    oneD[0].set_ydata(x[:frame,3])
    threeD[0].set_data_3d([0,x[frame,0]],[0,x[frame,1]],[0,x[frame,2]])
    return (oneD[0],threeD[0])

ani = pltAnim.FuncAnimation(fig=fig, func=update, frames=N_STEPS, interval=30)
ani.save("/home/exj5139/Downloads/abdomenAni.mp4")
plt.show()